<a href="https://colab.research.google.com/github/tsamarahhuwaida/mangrove-lai-skripsi/blob/main/notebooks/1.%20Batch_Process_Uncorrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
PIPELINE UNCORRECTED LAI — Konfigurasi Terpusat
=============================================================================
Alur:
  1. Float         : DN → float32 (÷10000)
  2. Masking Mangrove : crop + mask dengan shapefile mangrove
  3. Vegetation Indices : hitung NDVI, EVI, SAVI, ARVI
  4. Layerstack     : gabung band spektral + VI menjadi 8-band
  5. Masking NDWI   : terapkan mask air dari folder eksternal
  6. RF LAI         : prediksi LAI menggunakan model Random Forest

Cara pakai:
  1. Edit bagian CONFIG di bawah
  2. Jalankan seluruh script — semua tahap akan berjalan otomatis
=============================================================================
"""

# =============================================================================
# ██████████  KONFIGURASI TERPUSAT — edit bagian ini saja  ██████████████████
# =============================================================================
# ⚠️ Sesuaikan semua path di bawah dengan struktur Google Drive Anda.
#    Jalankan cell ini sekali sebelum menjalankan tahap lainnya.
# =============================================================================

import os

# ── Root dataset ──────────────────────────────────────────────────────────────
# Folder yang berisi file .tif mentah (integer DN, belum difloat)
DATASET_ROOT = "/content/drive/MyDrive/YOUR_PROJECT_FOLDER/imagery/raw"

# ── Input: shapefile masking mangrove ─────────────────────────────────────────
SHP_MANGROVE = "/content/drive/MyDrive/YOUR_PROJECT_FOLDER/shapefiles/mangrove_mask.shp"

# ── Input: folder mask NDWI eksternal ─────────────────────────────────────────
DIR_MASK_NDWI = "/content/drive/MyDrive/YOUR_PROJECT_FOLDER/ndwi_mask"

# ── Input: model Random Forest ────────────────────────────────────────────────
_RF_DIR      = "/content/drive/MyDrive/YOUR_PROJECT_FOLDER/model"
MODEL_PKL    = f"{_RF_DIR}/rf_lai_final_model.pkl"
FEATURES_PKL = f"{_RF_DIR}/selected_features.pkl"

# ── Parameter umum ────────────────────────────────────────────────────────────
YEARS        = list(range(2016, 2026))   # sesuaikan rentang tahun jika perlu
NODATA_VALUE = 0
CHUNK_SIZE   = 100_000

# ── Output: subfolder diturunkan otomatis (tidak perlu diubah) ────────────────
_DS = DATASET_ROOT.rstrip("/").split("/")[-1]

DIR_FLOAT    = f"{DATASET_ROOT}/{_DS}_Float"
DIR_MASKED   = f"{DATASET_ROOT}/{_DS}_Masked"
DIR_VI       = f"{DATASET_ROOT}/{_DS}_VI"
DIR_LAYSTACK = f"{DATASET_ROOT}/{_DS}_Laystack"
DIR_NDWI     = f"{DATASET_ROOT}/{_DS}_Masking_NDWI"
DIR_LAI      = f"{DATASET_ROOT}/{_DS}_LAI"

for d in [DIR_FLOAT, DIR_MASKED, DIR_VI, DIR_LAYSTACK, DIR_NDWI, DIR_LAI]:
    os.makedirs(d, exist_ok=True)

print("✅ Konfigurasi berhasil dimuat.")
print(f"   DATASET_ROOT : {DATASET_ROOT}")
print(f"   YEARS        : {YEARS[0]}–{YEARS[-1]}")
# =============================================================================
# END OF CONFIG
# =============================================================================

import os, re, glob, warnings
import numpy as np
import rasterio
from rasterio.mask import mask as rasterio_mask
from rasterio.crs import CRS
from rasterio.warp import reproject, Resampling
from pathlib import Path
import geopandas as gpd
import joblib
warnings.filterwarnings("ignore")

# Buat semua folder output
for d in [DIR_FLOAT, DIR_MASKED, DIR_VI, DIR_LAYSTACK, DIR_NDWI, DIR_LAI]:
    os.makedirs(d, exist_ok=True)


def section(title):
    print("\n" + "=" * 60)
    print(f"  {title}")
    print("=" * 60)


def build_year_map(folder, pattern=r"(19|20)\d{2}"):
    """Buat dict {tahun: path} berdasarkan 4-digit tahun dalam nama file."""
    year_map = {}
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.tif', '.tiff')):
            continue
        m = re.search(pattern, os.path.splitext(fname)[0])
        if m:
            yr = m.group(0)
            if yr not in year_map:
                year_map[yr] = os.path.join(folder, fname)
    return year_map


# =============================================================================
# TAHAP 1 — FLOAT
# =============================================================================
section("TAHAP 1 / 6 — Float (DN → float32 ÷ 10000)")

raw_files = sorted(Path(DIR_RAW).glob("*.tif"))
print(f"  Ditemukan {len(raw_files)} file di {DIR_RAW}\n")

for fpath in raw_files:
    out_name = fpath.stem[:4] + fpath.suffix
    out_path = Path(DIR_FLOAT) / out_name

    if out_path.exists():
        print(f"  ✓ skip (sudah ada): {out_name}")
        continue

    with rasterio.open(fpath) as src:
        data    = src.read().astype(np.float32) / 10000
        profile = src.profile.copy()
        profile.update(dtype='float32')

    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(data)

    print(f"  ✓ {fpath.name}  →  {out_name}")

print(f"\n  Output: {DIR_FLOAT}")


# =============================================================================
# TAHAP 2 — MASKING MANGROVE
# =============================================================================
section("TAHAP 2 / 6 — Masking Mangrove")

def reproject_shapes(shp_path, target_crs):
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None:
        return [g.__geo_interface__ for g in gdf.geometry if g is not None]
    gdf = gdf.to_crs(target_crs.to_wkt())
    return [g.__geo_interface__ for g in gdf.geometry if g is not None]


float_files = sorted(Path(DIR_FLOAT).glob("*.tif"))
print(f"  Ditemukan {len(float_files)} file di {DIR_FLOAT}\n")

for fpath in float_files:
    out_path = Path(DIR_MASKED) / (fpath.stem + "_masked.tif")

    if out_path.exists():
        print(f"  ✓ skip (sudah ada): {out_path.name}")
        continue

    print(f"  Processing: {fpath.name}")
    try:
        with rasterio.open(fpath) as src:
            shapes      = reproject_shapes(SHP_MANGROVE, src.crs)
            out_img, tf = rasterio_mask(src, shapes, crop=True,
                                        nodata=NODATA_VALUE, all_touched=True)
            meta = src.meta.copy()
            meta.update({"driver": "GTiff", "height": out_img.shape[1],
                         "width": out_img.shape[2], "transform": tf,
                         "nodata": NODATA_VALUE})

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(out_img)

        print(f"    ✅ Saved → {out_path.name}")
    except Exception as e:
        print(f"    ❌ Error: {e}")

print(f"\n  Output: {DIR_MASKED}")


# =============================================================================
# TAHAP 3 — VEGETATION INDICES
# =============================================================================
section("TAHAP 3 / 6 — Vegetation Indices (NDVI, EVI, SAVI, ARVI)")

def compute_ndvi(nir, red):
    d = nir + red
    return np.where(d != 0, (nir - red) / d, np.nan)

def compute_evi(nir, red, blue, G=2.5, C1=6, C2=7.5, L=1):
    d = nir + C1*red - C2*blue + L
    return np.where(d != 0, G * (nir - red) / d, np.nan)

def compute_savi(nir, red, L=0.5):
    d = nir + red + L
    return np.where(d != 0, (nir - red) * (1 + L) / d, np.nan)

def compute_arvi(nir, red, blue, y=1.0):
    rb = red - y * (blue - red)
    d  = nir + rb
    return np.where(d != 0, (nir - rb) / d, np.nan)


masked_files = sorted(Path(DIR_MASKED).glob("*.tif"))
print(f"  Ditemukan {len(masked_files)} file di {DIR_MASKED}\n")

for fpath in masked_files:
    out_path = Path(DIR_VI) / (fpath.stem + "_VI.tiff")

    if out_path.exists():
        print(f"  ✓ skip (sudah ada): {out_path.name}")
        continue

    print(f"  Processing: {fpath.name}")
    try:
        with rasterio.open(fpath) as src:
            blue    = src.read(1).astype(np.float32)
            red     = src.read(3).astype(np.float32)
            nir     = src.read(4).astype(np.float32)
            profile = src.profile.copy()

        ndvi = compute_ndvi(nir, red)
        evi  = compute_evi(nir, red, blue)
        savi = compute_savi(nir, red)
        arvi = compute_arvi(nir, red, blue)

        profile.update(dtype='float32', count=4, nodata=np.nan)

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(ndvi.astype(np.float32), 1)
            dst.write(evi.astype(np.float32),  2)
            dst.write(savi.astype(np.float32), 3)
            dst.write(arvi.astype(np.float32), 4)
            dst.update_tags(1, name='NDVI')
            dst.update_tags(2, name='EVI')
            dst.update_tags(3, name='SAVI')
            dst.update_tags(4, name='ARVI')

        print(f"    ✅ Saved → {out_path.name}")
    except Exception as e:
        print(f"    ❌ Error: {e}")

print(f"\n  Output: {DIR_VI}")


# =============================================================================
# TAHAP 4 — LAYERSTACK
# =============================================================================
section("TAHAP 4 / 6 — Layerstack (Masked + VI → 8 band)")

map_masked = build_year_map(DIR_MASKED)
map_vi     = build_year_map(DIR_VI)
matched    = sorted(set(map_masked) & set(map_vi))

print(f"  Masked : {len(map_masked)} file | VI : {len(map_vi)} file")
print(f"  Matched: {matched}\n")

for year in matched:
    out_path = os.path.join(DIR_LAYSTACK, f"{year}_stacked.tif")

    if os.path.exists(out_path):
        print(f"  ✓ skip (sudah ada): {year}_stacked.tif")
        continue

    path_a = map_masked[year]  # 4 band spektral
    path_b = map_vi[year]      # 4 band VI

    print(f"  Stacking {year}...")
    try:
        with rasterio.open(path_a) as src_a:
            meta          = src_a.meta.copy()
            data_a        = src_a.read().astype(np.float32)
            dst_crs       = src_a.crs
            dst_transform = src_a.transform
            dst_h, dst_w  = src_a.height, src_a.width

        with rasterio.open(path_b) as src_b:
            n_b = src_b.count
            if (src_b.height != dst_h or src_b.width != dst_w or
                    src_b.crs != dst_crs):
                data_b = np.zeros((n_b, dst_h, dst_w), dtype=np.float32)
                for i in range(n_b):
                    reproject(source=src_b.read(i+1).astype(np.float32),
                               destination=data_b[i],
                               src_transform=src_b.transform,
                               src_crs=src_b.crs,
                               dst_transform=dst_transform,
                               dst_crs=dst_crs,
                               resampling=Resampling.bilinear)
            else:
                data_b = src_b.read().astype(np.float32)

        stacked = np.concatenate([data_a, data_b], axis=0)
        meta.update({"dtype": "float32", "count": stacked.shape[0]})

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(stacked)

        print(f"    ✅ {year}_stacked.tif  ({stacked.shape[0]} bands)")
    except Exception as e:
        print(f"    ❌ Error: {e}")

print(f"\n  Output: {DIR_LAYSTACK}")


# =============================================================================
# TAHAP 5 — MASKING NDWI
# =============================================================================
section("TAHAP 5 / 6 — Masking NDWI (hapus piksel air)")

map_stack = build_year_map(DIR_LAYSTACK)
map_ndwi  = build_year_map(DIR_MASK_NDWI)
matched   = sorted(set(map_stack) & set(map_ndwi))

only_stack = sorted(set(map_stack) - set(map_ndwi))
print(f"  Laystack : {len(map_stack)} file | Mask NDWI : {len(map_ndwi)} file")
print(f"  Matched  : {matched}")
if only_stack:
    print(f"  ⚠️  Tidak ada mask NDWI untuk: {only_stack} — akan dilewati\n")

for year in matched:
    orig_path = map_stack[year]
    mask_path = map_ndwi[year]
    orig_fname = os.path.basename(orig_path)
    out_path  = os.path.join(DIR_NDWI,
                             os.path.splitext(orig_fname)[0] + "_masked.tif")

    if os.path.exists(out_path):
        print(f"  ✓ skip (sudah ada): {os.path.basename(out_path)}")
        continue

    print(f"  Masking {year}...")
    try:
        with rasterio.open(mask_path) as msrc:
            mask_raw = msrc.read(1).astype(np.float32)
            with rasterio.open(orig_path) as src:
                meta   = src.meta.copy()
                data   = src.read().astype(np.float32)
                dst_h, dst_w = src.height, src.width
                dst_crs = src.crs
                dst_tf  = src.transform

            if (msrc.height != dst_h or msrc.width != dst_w or
                    msrc.crs != dst_crs):
                aligned = np.zeros((dst_h, dst_w), dtype=np.float32)
                reproject(source=mask_raw, destination=aligned,
                          src_transform=msrc.transform, src_crs=msrc.crs,
                          dst_transform=dst_tf, dst_crs=dst_crs,
                          resampling=Resampling.nearest)
                mask_raw = aligned

        invalid = mask_raw == -9999
        for b in range(data.shape[0]):
            data[b][invalid] = 0
        data[data == -9999] = 0

        meta.update({"dtype": "float32", "nodata": 0})
        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(data)

        pct = 100 * invalid.sum() / invalid.size
        print(f"    ✅ {os.path.basename(out_path)}  "
              f"(masked {invalid.sum():,} px = {pct:.1f}%)")
    except Exception as e:
        print(f"    ❌ Error: {e}")

print(f"\n  Output: {DIR_NDWI}")


# =============================================================================
# TAHAP 6 — RF LAI PREDICTION
# =============================================================================
section("TAHAP 6 / 6 — RF LAI Prediction")

BAND_MAP = {
    "Band_1": 1, "Band_2": 2, "Band_3": 3, "Band_4": 4,
    "NDVI":   5, "EVI":   6,  "SAVI":  7,  "ARVI ":  8,
}
VALID_RANGE  = (0, 1)
OUTPUT_NODATA = -9999

print("  Loading model...")
rf_model = joblib.load(MODEL_PKL)
selected_features = (joblib.load(FEATURES_PKL)
                     if FEATURES_PKL and os.path.exists(FEATURES_PKL)
                     else list(BAND_MAP.keys()))
print(f"  Trees    : {rf_model.n_estimators}")
print(f"  Features : {selected_features}\n")

all_tifs = sorted(glob.glob(os.path.join(DIR_NDWI, "*.tif")))
image_queue = []
for tif in all_tifs:
    fname = os.path.basename(tif)
    m = re.search(r"(20\d{2})", fname)
    if m and int(m.group(1)) in YEARS:
        image_queue.append((m.group(1), tif, fname))

print(f"  Ditemukan {len(image_queue)} gambar untuk diproses\n")

results, skipped = [], []

for i, (year, path, fname) in enumerate(image_queue, 1):
    out_path = os.path.join(DIR_LAI, f"{year}_lai_predicted.tif")

    if os.path.exists(out_path):
        print(f"  ✓ skip (sudah ada): {year}_lai_predicted.tif")
        continue

    print(f"\n  [{i}/{len(image_queue)}] {year} — {fname}")

    try:
        with rasterio.open(path) as src:
            profile = src.profile
            h, w    = src.height, src.width
            nodata  = src.nodata
            band_data = {f: src.read(BAND_MAP[f]).astype(float)
                         for f in selected_features if f in BAND_MAP}

        # Mask valid pixels
        valid = np.ones((h, w), dtype=bool)
        for arr in band_data.values():
            valid &= (arr != NODATA_VALUE)
        if nodata is not None:
            for arr in band_data.values():
                valid &= (arr != nodata)
        for arr in band_data.values():
            valid &= (arr >= VALID_RANGE[0]) & (arr <= VALID_RANGE[1])
            valid &= np.isfinite(arr)

        n_valid = valid.sum()
        print(f"    Valid pixels: {n_valid:,} / {h*w:,} "
              f"({100*n_valid/(h*w):.1f}%)")

        if n_valid == 0:
            print(f"    ⚠️  Tidak ada piksel valid — skip")
            skipped.append(fname)
            continue

        # Predict in chunks
        feat_matrix = np.stack([band_data[f][valid] for f in selected_features],
                               axis=1)
        lai_flat = np.zeros(n_valid, dtype=float)
        for s in range(0, n_valid, CHUNK_SIZE):
            e = min(s + CHUNK_SIZE, n_valid)
            lai_flat[s:e] = rf_model.predict(feat_matrix[s:e])
            print(f"    {e:,}/{n_valid:,} px ({100*e/n_valid:.0f}%)", end="\r")

        print(f"\n    LAI — mean:{lai_flat.mean():.4f}  "
              f"min:{lai_flat.min():.4f}  max:{lai_flat.max():.4f}")

        # Reconstruct & save
        lai_img = np.full((h, w), np.nan, dtype=float)
        lai_img[valid] = lai_flat
        lai_out = lai_img.astype("float32")
        lai_out[np.isnan(lai_img)] = OUTPUT_NODATA

        out_profile = profile.copy()
        out_profile.update({"count": 1, "dtype": "float32",
                            "nodata": OUTPUT_NODATA, "compress": "lzw"})

        with rasterio.open(out_path, "w", **out_profile) as dst:
            dst.write(lai_out, 1)

        print(f"    ✅ Saved: {year}_lai_predicted.tif")
        results.append({"year": year, "fname": fname,
                        "n_valid": n_valid,
                        "mean": lai_flat.mean(), "min": lai_flat.min(),
                        "max": lai_flat.max()})
    except Exception as e:
        print(f"    ❌ Error: {e}")
        skipped.append(fname)


# =============================================================================
# RINGKASAN AKHIR
# =============================================================================
section("SELESAI — Ringkasan Pipeline")

print(f"\n  Folder output:")
print(f"    Float        : {DIR_FLOAT}")
print(f"    Masked       : {DIR_MASKED}")
print(f"    VI           : {DIR_VI}")
print(f"    Laystack     : {DIR_LAYSTACK}")
print(f"    NDWI masked  : {DIR_NDWI}")
print(f"    LAI result   : {DIR_LAI}")

if results:
    print(f"\n  {'Tahun':<8} {'Valid Px':>10} {'Mean LAI':>10} "
          f"{'Min':>8} {'Max':>8}")
    print("  " + "-" * 50)
    for r in results:
        print(f"  {r['year']:<8} {r['n_valid']:>10,} {r['mean']:>10.4f} "
              f"{r['min']:>8.4f} {r['max']:>8.4f}")

print(f"\n  ✅ Berhasil   : {len(results)} gambar")
if skipped:
    print(f"  ⚠️  Dilewati  : {len(skipped)} gambar — {skipped}")
print("=" * 60)


  TAHAP 1 / 6 — Float (DN → float32 ÷ 10000)
  Ditemukan 10 file di /content/drive/MyDrive/01-SKRIPSI PIF LAI CMC/04. BAB 4 - HASIL/04.01. PIF/04.01.01. Citra Mentah/01. Pengelompokkan Dataset/Dataset 6_copy2

  ✓ skip (sudah ada): 2016.tif
  ✓ skip (sudah ada): 2017.tif
  ✓ skip (sudah ada): 2018.tif
  ✓ skip (sudah ada): 2019.tif
  ✓ skip (sudah ada): 2020.tif
  ✓ skip (sudah ada): 2021.tif
  ✓ skip (sudah ada): 2022.tif
  ✓ skip (sudah ada): 2023.tif
  ✓ skip (sudah ada): 2024.tif
  ✓ skip (sudah ada): 2025.tif

  Output: /content/drive/MyDrive/01-SKRIPSI PIF LAI CMC/04. BAB 4 - HASIL/04.01. PIF/04.01.01. Citra Mentah/01. Pengelompokkan Dataset/Dataset 6_copy2/Dataset 6_copy2_Float

  TAHAP 2 / 6 — Masking Mangrove
  Ditemukan 10 file di /content/drive/MyDrive/01-SKRIPSI PIF LAI CMC/04. BAB 4 - HASIL/04.01. PIF/04.01.01. Citra Mentah/01. Pengelompokkan Dataset/Dataset 6_copy2/Dataset 6_copy2_Float

  ✓ skip (sudah ada): 2016_masked.tif
  ✓ skip (sudah ada): 2017_masked.tif
  ✓ skip

    62,985/62,985 px (100%)
    LAI — mean:0.7001  min:0.0000  max:2.0726
    ✅ Saved: 2016_lai_predicted.tif

  [2/10] 2017 — 2017_stacked_masked.tif
    Valid pixels: 63,665 / 247,044 (25.8%)


    63,665/63,665 px (100%)
    LAI — mean:0.8186  min:0.0000  max:2.0726
    ✅ Saved: 2017_lai_predicted.tif

  [3/10] 2018 — 2018_stacked_masked.tif
    Valid pixels: 67,015 / 247,044 (27.1%)


    67,015/67,015 px (100%)
    LAI — mean:0.5848  min:0.0000  max:2.0388
    ✅ Saved: 2018_lai_predicted.tif

  [4/10] 2019 — 2019_stacked_masked.tif
    Valid pixels: 64,188 / 247,044 (26.0%)


    64,188/64,188 px (100%)
    LAI — mean:0.6233  min:0.0000  max:2.0807
    ✅ Saved: 2019_lai_predicted.tif

  [5/10] 2020 — 2020_stacked_masked.tif
    Valid pixels: 65,975 / 247,044 (26.7%)


...

    65,041/65,041 px (100%)
    LAI — mean:1.5049  min:0.1228  max:2.2409
    ✅ Saved: 2021_lai_predicted.tif

  [7/10] 2022 — 2022_stacked_masked.tif
    Valid pixels: 67,975 / 247,044 (27.5%)


    67,975/67,975 px (100%)
    LAI — mean:1.5698  min:0.0000  max:2.1469
    ✅ Saved: 2022_lai_predicted.tif

  [8/10] 2023 — 2023_stacked_masked.tif
    Valid pixels: 67,916 / 247,044 (27.5%)


    67,916/67,916 px (100%)
    LAI — mean:1.4430  min:0.0000  max:2.0726
    ✅ Saved: 2023_lai_predicted.tif

  [9/10] 2024 — 2024_stacked_masked.tif
    Valid pixels: 68,919 / 247,044 (27.9%)


    68,919/68,919 px (100%)
    LAI — mean:0.9459  min:0.0000  max:2.0709
    ✅ Saved: 2024_lai_predicted.tif

  [10/10] 2025 — 2025_stacked_masked.tif
    Valid pixels: 69,096 / 247,044 (28.0%)


    69,096/69,096 px (100%)
    LAI — mean:1.6382  min:0.0000  max:2.2409
    ✅ Saved: 2025_lai_predicted.tif

  SELESAI — Ringkasan Pipeline

  Folder output:
    Float        : /content/drive/MyDrive/01-SKRIPSI PIF LAI CMC/04. BAB 4 - HASIL/04.01. PIF/04.01.01. Citra Mentah/01. Pengelompokkan Dataset/Dataset 6_copy2/Dataset 6_copy2_Float
    Masked       : /content/drive/MyDrive/01-SKRIPSI PIF LAI CMC/04. BAB 4 - HASIL/04.01. PIF/04.01.01. Citra Mentah/01. Pengelompokkan Dataset/Dataset 6_copy2/Dataset 6_copy2_Masked
    VI           : /content/drive/MyDrive/01-SKRIPSI PIF LAI CMC/04. BAB 4 - HASIL/04.01. PIF/04.01.01. Citra Mentah/01. Pengelompokkan Dataset/Dataset 6_copy2/Dataset 6_copy2_VI
    Laystack     : /content/drive/MyDrive/01-SKRIPSI PIF LAI CMC/04. BAB 4 - HASIL/04.01. PIF/04.01.01. Citra Mentah/01. Pengelompokkan Dataset/Dataset 6_copy2/Dataset 6_copy2_Laystack
    NDWI masked  : /content/drive/MyDrive/01-SKRIPSI PIF LAI CMC/04. BAB 4 - HASIL/04.01. PIF/04.01.01. Citra M